# agent

> Slash-command REPL over a Lean project + vLLM endpoint. It now builds a compact Lean project context automatically, loads repo-local instructions such as `.gcd/agent.md`, finds `sorry` declarations via Lean LSP, sends context-aware requests to vLLM, splices candidates, runs `lake build`, and reverts on failure.


In [ ]:
#| default_exp agent

The implementation lives in [`slurm_ops/agent.py`](../slurm_ops/agent.py).
Because the agent code wraps Lean project inspection, a Lean LSP client, a vLLM
HTTP client, file splicing, and a `cmd.Cmd` REPL, the source of truth is the
.py file rather than this notebook. The notebook is here for nbdev parity and
to document the public surface.

Public functions (matches `slurm_ops.agent.__all__`):

- `load_env_file(path)` — KEY=value loader, `os.environ.setdefault` semantics.
- `load_project_instructions(project_root, explicit_path)` — load `.gcd/agent.md` or another project instruction file.
- `inspect_project(project_root)` — collect Lake metadata, toolchain, deps, source roots, git status, README excerpt, and instruction text.
- `render_project_context(ctx)` — compact prompt section for the model.
- `find_sorrys(project_root)` — list `SorryHit` for every sorry the Lean compiler flags, including file imports and local context.
- `ask_for_proof(decl, sig, ctx, project_context=...)` — OpenAI chat request for a replacement declaration; returns a `Proposal`.
- `ask_project_question(question, project_context=...)` — OpenAI chat request for free-form project Q&A.
- `splice(text, hit, replacement)` — substitute the enclosing declaration in `text`.
- `lake_build(project_root, module)` — `(rc, stdout, stderr)`.
- `AgentShell` — `cmd.Cmd` subclass implementing chat/context/instructions/list/show/ask/try/build/env/quit.
- `run_loop(project_root)` — inspect project, instantiate `AgentShell`, and `cmdloop()`.
- `main(argv)` — CLI entrypoint for `agent/bin/gcd-agent`.

Persistent project guidance should live in the Lean project as `.gcd/agent.md`.
Session-only guidance can be passed with `gcd-agent --context "..."` or added
inside the REPL with `instructions TEXT`.


In [ ]:
#| hide
# nbdev_export() is intentionally NOT called from this notebook because
# slurm_ops/agent.py is the authoritative source. If you re-export from
# this notebook you'll overwrite the real implementation with a stub.